# Module 4 - RAG pipeline

## Step 1 : Data Prepration & EDA

In [1]:
from datasets import load_dataset
import pandas as pd 
import numpy as np
import re
import os
import joblib
from tqdm.auto import tqdm
from sentence_transformers import SentenceTransformer
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics.pairwise import cosine_similarity
import nltk
from nltk.tokenize import sent_tokenize
import torch

c:\Users\win11\anaconda3\envs\rag-chatbot\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Login using e.g. `huggingface-cli login` to access this dataset
ds = load_dataset("Amod/mental_health_counseling_conversations")
df = pd.DataFrame(ds['train'])
df.head()

,Context,Response
0,I'm going through some things with my feelings...,"If everyone thinks you're worthless, then mayb..."
1,I'm going through some things with my feelings...,"Hello, and thank you for your question and see..."
2,I'm going through some things with my feelings...,First thing I'd suggest is getting the sleep y...
3,I'm going through some things with my feelings...,Therapy is essential for those that are feelin...
4,I'm going through some things with my feelings...,I first want to let you know that you are not ...


In [3]:
print("Before cleaning:\n")
# checking the number of Context & Response
print(f'Total Context & Response: {len(df)}')
print("="*50)

# checking the null values
print(f"Total null values: \n{df.isnull().sum()}")
print("="*50)

# checking duplicates
print(f"Duplicates rows: {df.duplicated().sum()}")
print(f"Duplicates Response: {df.duplicated(subset='Response').sum()}")
print(f"Duplicates Context: {df.duplicated(subset='Context').sum()}")
print("="*50)

Before cleaning:

Total Context & Response: 3512
Total null values: 
Context     0
Response    0
dtype: int64
Duplicates rows: 760
Duplicates Response: 1032
Duplicates Context: 2517


In [5]:
def clean_text(text):
    '''Clean text by removing URLs, HTML tags, extra spaces and newlines.'''
    text = str(text)

    # Remove HTML comments
    text = re.sub(r'<!--.*?-->', '', text, flags=re.DOTALL)
    # Remove quoted attribute values:  src="..."  or  src='...'  (with or without closing quote)
    text = re.sub(r'\w+\s*=\s*["\'][^"\']*["\']?', '', text)
    # Remove unquoted attribute values:  src=...
    text = re.sub(r'\w+\s*=\s*[^\s>"\']+', '', text)
    # Remove remaining tag shell  <...>  or  <...  (no closing >)
    text = re.sub(r'<[^>]*>?', '', text)

    # Remove URLs
    text = re.sub(r'http[s]?://\S+', '', text)
    text = re.sub(r'www\.\S+', '', text)
    text = re.sub(r'\b\S+\.(com|org|net|ca|io|pdf|html|htm)\S*', '', text)

    # Clean whitespace
    text = text.replace('\n', ' ')
    return re.sub(r'\s+', ' ', text).strip()

df['Context'] = df['Context'].apply(clean_text)
df['Response'] = df['Response'].apply(clean_text)

# Delete rows with empty Context or Response, drop short rows and duplicates
df = df[(df['Context'] != "") & (df['Response'] != "")]
df = df[df['Context'].str.len() >= 15]
df = df[df['Response'].str.len() >= 15]
df = df.drop_duplicates()

In [6]:
print("After cleaning:\n")

# checking the number of Context & Response
print(f'Total Context & Response: {len(df)}')
print("="*50)

# checking the null values
print(f"Total null values: \n{df.isnull().sum()}")
print("="*50)

# checking duplicates
print(f"Duplicates rows: {df.duplicated().sum()}")
print(f"Duplicates Response: {df.duplicated(subset='Response').sum()}")
print(f"Duplicates Context: {df.duplicated(subset='Context').sum()}")
print("="*50)

After cleaning:

Total Context & Response: 2021
Total null values: 
Context     0
Response    0
dtype: int64
Duplicates rows: 0
Duplicates Response: 1
Duplicates Context: 1191


In [7]:
# There are some 'Contexts' that have multiple 'Responses'.
dup_context = df.groupby('Context')['Response'].nunique().sort_values(ascending=False)
dup_context[dup_context > 1]

Context
I have so many issues to address. I have a history of sexual abuse, I’m a breast cancer survivor and I am a lifetime insomniac. I have a long history of depression and I’m beginning to have anxiety. I have low self esteem but I’ve been happily married for almost 35 years. I’ve never had counseling about any of this. Do I have too many issues to address in counseling?                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                               

In [8]:
conflicts = (
    df.groupby('Context')
      .filter(lambda x: x['Response'].nunique() > 1)
      .sort_values('Context')
)

with open("conflicts_output.txt", "w", encoding="utf-8") as f:
    for context, group in conflicts.groupby('Context'):
        f.write("=" * 80 + "\n")
        f.write("CONTEXT:\n\n")
        f.write(str(context) + "\n")

        f.write("\nRESPONSES:\n\n")
        for i, response in enumerate(group['Response'].unique(), 1):
            f.write(f"{i}. {response}\n\n")

print("Saved to conflicts_output.txt")

Saved to conflicts_output.txt


In [9]:
# saving the cleaned dataframe to a csv file in data folder
df.to_csv('data/df_cleaned.csv', index=False)

### Handle mutiple Response of the same Context

In [10]:
# Load embedding model
model = SentenceTransformer('all-MiniLM-L6-v2')

# Precompute embeddings for all unique responses and cache to disk
emb_cache_path = os.path.join('data', 'response_embeddings.joblib')
responses = df['Response'].unique().tolist()

if os.path.exists(emb_cache_path):
    emb_lookup = joblib.load(emb_cache_path)
    print(f'Loaded cached embeddings for {len(emb_lookup)} responses')
else:
    print(f'Computing embeddings for {len(responses)} unique responses...')
    emb_array = model.encode(
        responses,
        convert_to_numpy=True,
        normalize_embeddings=True,
        batch_size=64,
        show_progress_bar=True
    )
    emb_lookup = {r: emb_array[i] for i, r in enumerate(responses)}
    os.makedirs(os.path.dirname(emb_cache_path), exist_ok=True)
    joblib.dump(emb_lookup, emb_cache_path)
    print(f'Saved embeddings cache to {emb_cache_path}')

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3961.50it/s]


Loaded cached embeddings for 2022 responses


#### Clustering-based Summarization

In [11]:
def semantic_clustering_summarize(responses, emb_lookup, similarity_threshold=0.75):
    """ Cluster similar responses and return a representative response for each cluster.
    Args:
        responses (list): List of responses to cluster.
        emb_lookup (dict): Mapping response -> embedding (numpy array).
        similarity_threshold (float): Threshold for cosine similarity to consider responses as similar.
    Returns:
        list: List of representative responses for each cluster.
    """
    # if there is only one response
    if len(responses) <= 1:
        return responses

    # Build embeddings matrix from lookup; fall back to encoding missing responses
    missing = [r for r in responses if r not in emb_lookup]
    if missing:
        # encode missing ones on-the-fly
        missing_embs = model.encode(missing, convert_to_numpy=True, normalize_embeddings=True, batch_size=64, show_progress_bar=False)
        for i, r in enumerate(missing):
            emb_lookup[r] = missing_embs[i]

    embeddings = np.vstack([emb_lookup[r] for r in responses])

    # Clustering
    clustering_model = AgglomerativeClustering(n_clusters=None, metric='cosine', linkage='average', distance_threshold=1 - similarity_threshold)
    labels = clustering_model.fit_predict(embeddings)

    # Pick representative response
    unique_views = []

    for cluster_id in np.unique(labels):
        cluster_indices = np.where(labels == cluster_id)[0]
        cluster_embeddings = embeddings[cluster_indices]

        # centroid of cluster
        centroid = cluster_embeddings.mean(axis=0)
        # normalize centroid to unit length (avoid zero division)
        norm = np.linalg.norm(centroid)
        if norm > 0:
            centroid = centroid / (norm + 1e-12)

        # similarity to centroid
        similarities = cosine_similarity(cluster_embeddings, centroid.reshape(1, -1)).flatten()

        # best representative
        best_local_idx = np.argmax(similarities)
        best_idx = cluster_indices[best_local_idx]
        unique_views.append(responses[best_idx])

    return unique_views

In [12]:
final_data = []

# Process each Context (with progress bar)
for context, group in tqdm(df.groupby('Context'), total=df['Context'].nunique()):
    raw_responses = group['Response'].unique().tolist()
    summarized_views = semantic_clustering_summarize(raw_responses, emb_lookup, similarity_threshold=0.75)

    # Guard against empty summarization
    if not summarized_views:
        summarized_views = raw_responses[:1]

    # Format responses
    formatted_responses = (
        "\n".join([f"• {r}" for r in summarized_views])
        if len(summarized_views) > 1
        else summarized_views[0])

    final_data.append({'Context': context, 'Responses': formatted_responses})

# Save final dataframe
summarized_df = pd.DataFrame(final_data)
out_path = os.path.join('data', 'semantic_clustered_rag.csv')
summarized_df.to_csv(out_path, index=False)
print(f'Saved summarized dataframe to {out_path}')
summarized_df.head()

100%|██████████| 830/830 [00:08<00:00, 100.05it/s]


Saved summarized dataframe to data\semantic_clustered_rag.csv


,Context,Responses
0,A few nights ago I talked to this girl I know ...,Hey! It takes a lot of courage to share your f...
1,A few years ago I was making love to my wife w...,• First step always is to do a medical rule ou...
2,A friend of mine taking psychology advised I g...,I admire your courage for stating your view ab...
3,A girl and I were madly in love. We dated for ...,"Hi Boise, I'm sorry that you've lost this love..."
4,"A lot of times, I avoid situations where I am ...",• Why not accept and tolerate that you natural...


## Step 2 : Chunking

In [13]:
# Chunking Config 
CHUNK_SIZE_CHARS = 1500   # ~375 tokens per chunk
OVERLAP_CHARS    = 150    # chars carried over between chunks
BATCH_SIZE       = 64
MODEL_NAME       = 'all-MiniLM-L6-v2'

OUT_CHUNKS = os.path.join('data', 'chunks.parquet')
OUT_EMB    = os.path.join('artifacts', 'chunk_embeddings.joblib')
OUT_META   = os.path.join('artifacts', 'index_metadata.joblib')

print('Chunking config ready.')
print(f'  Chunk size : {CHUNK_SIZE_CHARS} chars')
print(f'  Overlap    : {OVERLAP_CHARS} chars')
print(f'  Model      : {MODEL_NAME}')

Chunking config ready.
  Chunk size : 1500 chars
  Overlap    : 150 chars
  Model      : all-MiniLM-L6-v2


In [14]:
def split_bullets(text):
    """
    Split a Responses field into individual therapist answers.
    Handles bullet markers: •  -  *  1.  and newlines.
    """
    if not text:
        return []
    parts = re.split(r'\n+', str(text))
    bullets = []
    for p in parts:
        s = p.strip()
        if not s:
            continue
        s = re.sub(r'^\s*(?:•|\-|\*|\d+\.)\s*', '', s)
        s = s.strip()
        if s:
            bullets.append(s)
    return bullets


def chunk_text(text, size=500, overlap=100):
    text = str(text).strip()
    if not text:
        return []
    if len(text) <= size:
        return [text]
    
    # Better sentence splitting. Preserves sentence boundaries and allows for more natural chunks.
    sentences = sent_tokenize(text)
    chunks = []
    current_chunk = ""
    for sentence in sentences:
        # If adding sentence exceeds chunk size
        if len(current_chunk) + len(sentence) + 1 > size:
            chunks.append(current_chunk.strip())
            # Add overlap from previous chunk
            overlap_text = current_chunk[-overlap:] if overlap > 0 else ""
            current_chunk = overlap_text + " " + sentence
        else:
            current_chunk += " " + sentence
    # Add last chunk
    if current_chunk.strip():
        chunks.append(current_chunk.strip())
    return chunks


In [15]:
# Load the semantic-clustered output from Step 1
chunking_df = pd.read_csv(os.path.join('data', 'semantic_clustered_rag.csv'))
print(f'Loaded {len(chunking_df):,} rows')
print(f'Columns: {chunking_df.columns.tolist()}')
chunking_df.head(3)

Loaded 830 rows
Columns: ['Context', 'Responses']


,Context,Responses
0,A few nights ago I talked to this girl I know ...,Hey! It takes a lot of courage to share your f...
1,A few years ago I was making love to my wife w...,• First step always is to do a medical rule ou...
2,A friend of mine taking psychology advised I g...,I admire your courage for stating your view ab...


In [16]:
records = []

for i, row in tqdm(chunking_df.iterrows(), total=len(chunking_df), desc='Chunking rows'):
    question = str(row.get('Context', row.iloc[0])).strip()

    # Support both 'Responses' and 'Response' column names
    responses_field = row.get('Responses') if 'Responses' in chunking_df.columns else row.get('Response')

    bullets = split_bullets(responses_field) if responses_field else []
    if not bullets:
        bullets = [responses_field] if responses_field else [question]

    for bi, bullet in enumerate(bullets):
        qa_text = f'Q: {question}\nA: {bullet}'
        for ci, chunk in enumerate(chunk_text(qa_text)):
            records.append({
                'context_id'       : int(i),
                'bullet_index'     : int(bi),
                'chunk_index'      : int(ci),
                'original_response': bullet,
                'text'             : chunk,
            })

chunks_df = pd.DataFrame(records)
print(f'Original rows : {len(chunking_df):,}')
print(f'Total chunks  : {len(chunks_df):,}')
chunks_df.head(3)

Chunking rows: 100%|██████████| 830/830 [00:00<00:00, 973.04it/s] 

Original rows : 830
Total chunks  : 6,680


,context_id,bullet_index,chunk_index,original_response,text
0,0,0,0,Hey! It takes a lot of courage to share your f...,Q: A few nights ago I talked to this girl I kn...
1,0,0,1,Hey! It takes a lot of courage to share your f...,"about her, but I leave to go back to college i..."
2,0,0,2,Hey! It takes a lot of courage to share your f...,lucky to meet someone who makes you feel safe ...


In [17]:
chunks_df['char_len'] = chunks_df['text'].str.len()

print('Chunk character length stats:')
print(chunks_df['char_len'].describe().round(1))
print()
print(f'Chunks under 500 chars  : {(chunks_df["char_len"] < 500).sum():,}')
print(f'Chunks 500–1500 chars   : {((chunks_df["char_len"] >= 500) & (chunks_df["char_len"] <= 1500)).sum():,}')
print(f'Chunks over 1500 chars  : {(chunks_df["char_len"] > 1500).sum():,}')

Chunk character length stats:
count    6680.0
mean      413.0
std       128.1
min         0.0
25%       351.0
50%       427.0
75%       472.0
max      1776.0
Name: char_len, dtype: float64

Chunks under 500 chars  : 6,210
Chunks 500–1500 chars   : 468
Chunks over 1500 chars  : 2


In [18]:
# Save chunks to Parquet for efficient storage and later embedding
os.makedirs('data', exist_ok=True)
chunks_df.to_parquet(OUT_CHUNKS, index=False)
print(f'Saved {len(chunks_df):,} chunks → {OUT_CHUNKS}')

Saved 6,680 chunks → data\chunks.parquet


In [19]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Loading embedding model: {MODEL_NAME} on {device}...')
embed_model = SentenceTransformer(MODEL_NAME, device=device)

texts    = chunks_df['text'].tolist()
emb_list = []

for i in tqdm(range(0, len(texts), BATCH_SIZE), desc='Embedding batches'):
    batch = texts[i : i + BATCH_SIZE]
    emb   = embed_model.encode(
        batch,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False,
    )
    emb_list.append(emb)

embeddings = np.vstack(emb_list)
print(f'Embeddings shape: {embeddings.shape}')  # (n_chunks, 384)

Loading embedding model: all-MiniLM-L6-v2 on cpu...


Embedding batches: 100%|██████████| 105/105 [04:29<00:00,  2.57s/it]

Embeddings shape: (6680, 384)


In [20]:
joblib.dump(embeddings,  OUT_EMB)
joblib.dump(chunks_df.to_dict(orient='records'), OUT_META)

print(f'Saved embeddings → {OUT_EMB}   shape={embeddings.shape}')
print(f'Saved metadata   → {OUT_META}  records={len(chunks_df):,}')

Saved embeddings → artifacts\chunk_embeddings.joblib   shape=(6680, 384)
Saved metadata   → artifacts\index_metadata.joblib  records=6,680


In [21]:
# Quick check: retrieve nearest chunk to a sample query
from sklearn.metrics.pairwise import cosine_similarity as cos_sim

sample_query = 'I feel hopeless and do not know what to do'
q_emb = embed_model.encode([sample_query], normalize_embeddings=True)
scores = cos_sim(q_emb, embeddings)[0]
top_idx = scores.argsort()[::-1][:3]

print(f'Query: "{sample_query}"\n')
for rank, idx in enumerate(top_idx, 1):
    print(f'--- Rank {rank}  (score={scores[idx]:.4f}) ---')
    print(chunks_df.iloc[idx]['text'][:300])
    print()

Query: "I feel hopeless and do not know what to do"

--- Rank 1  (score=0.6172) ---
you are feeling like things are hopeless and out of control and you're not sure what to do about it. If you can find a competent therapist to work with, together you may be able to come up with some strategies for alleviating the overwhelming distress that you are experiencing and gain some insight 

--- Rank 2  (score=0.6017) ---
everything the books say I should do, but I don't feel any different. I just don't know what to do. A: Sounds like you need closure. I'm sure your doing your best to overcome this feeling but seem to be struggling with your own happiness. Trust God no one else. Give this some time and don't close yo

--- Rank 3  (score=0.5616) ---
ut down. I haven't felt that sense of comfort and happiness with myself since everything fell apart. I'm scared to because I don't want it to be taken away from me again. I feel like ever lesson I learn only last a day. I just don't know what to do. 